# cube3d on Colab (free T4)

Generates a `.glb` from a text prompt + bbox using Roblox's cube3d model on a free Colab T4 GPU, then downloads it to your machine.

**Setup once:**
1. Open this notebook at https://colab.research.google.com — upload it via *File > Upload notebook*.
2. Runtime > Change runtime type > **T4 GPU**.
3. Run all cells top-to-bottom.

**Each generation:** edit the `PROMPT` / `BBOX` in the *Generate* cell and re-run that cell + the *Download* cell.

To use it from VS Code instead of the Colab UI, run the **Optional: expose as a Jupyter server** cell at the bottom and connect VS Code's Jupyter extension to the printed URL.

## 1. Check GPU

In [ ]:
!nvidia-smi | head -n 15

## 2. Install cube3d

Clones Roblox/cube and installs in editable mode. Takes 2-3 min the first time.

In [ ]:
import os, sys, subprocess
if not os.path.isdir('/content/cube'):
    !git clone --depth=1 https://github.com/Roblox/cube.git /content/cube
%cd /content/cube
!pip install -q -e .
!pip install -q trimesh

## 3. Authenticate with HuggingFace

cube3d weights are gated. You need an HF account that has accepted the model card at https://huggingface.co/Roblox/cube3d-v0.5.

Paste your HF token (get one at https://huggingface.co/settings/tokens — *read* scope is enough). It is **not** saved anywhere persistent.

In [ ]:
from huggingface_hub import login, snapshot_download
import getpass
tok = getpass.getpass('HF token: ')
login(token=tok)
weights_dir = snapshot_download(repo_id='Roblox/cube3d-v0.5')
print('weights:', weights_dir)

## 4. Load the model

In [ ]:
import torch, os
from cube3d.inference.engine import EngineFast, Engine

config_path = '/content/cube/cube3d/configs/open_model_v0.5.yaml'
gpt_ckpt    = os.path.join(weights_dir, 'shape_gpt.safetensors')
shape_ckpt  = os.path.join(weights_dir, 'shape_tokenizer.safetensors')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EngineCls = EngineFast if device.type == 'cuda' else Engine
engine = EngineCls(config_path, gpt_ckpt, shape_ckpt, device=device)
print('engine ready on', device)

## 5. Generate

Edit `PROMPT` and `BBOX`, then run.

`BBOX` is in cube3d's normalized 0.1-2.0 range. For Roblox accessories use your existing helper to convert from studs; for full-body characters `(0.889, 1.0, 0.444)` ≈ chibi proportions (chunky 4-stud depth).

In [ ]:
import time, trimesh, os, gc, torch

torch.cuda.empty_cache()
gc.collect()
print('free before:', torch.cuda.mem_get_info()[0] / 1e9, 'GB')

PROMPT = 'cute chibi anthropomorphic shark character, T-pose, full body humanoid, big head, simple geometric shapes, friendly'
BBOX   = (0.889, 1.0, 0.444)
OUT    = f'/content/out_{int(time.time())}.glb'

t0 = time.time()
result = engine.t2s(
    [PROMPT],
    use_kv_cache=True,
    resolution_base=8.0,
    chunk_size=20000,
    bounding_box_xyz=BBOX,
)
verts, faces = result[0][0], result[0][1]
trimesh.Trimesh(vertices=verts, faces=faces).export(OUT)
print(f'wrote {OUT} in {time.time()-t0:.1f}s')
print('verts:', len(verts), 'faces:', len(faces))

## 6. Download to your machine

Triggers a browser download of the `.glb`. Drop it into `runs/` in your local repo, then run `robloxchars autorig` (or `gen-and-rig` with `--skip-gen --input <path>` if you wire that flag in).

In [ ]:
from google.colab import files
files.download(OUT)

---
## Optional: expose this runtime as a Jupyter server for VS Code

Run the cell below to start a Jupyter server inside Colab and open a public tunnel via cloudflared. Then in VS Code:

1. Cmd/Ctrl-Shift-P > **Jupyter: Specify Jupyter Server for Connections**
2. Pick **Existing** and paste the printed `https://....trycloudflare.com/?token=...` URL.
3. Open any `.ipynb` in VS Code and pick that remote as the kernel.

Now `engine` stays loaded in the Colab GPU; your VS Code cells execute on T4. Colab idle-disconnects after ~90 min of no activity, so keep a cell running or re-establish.

Note: trycloudflare URLs are public-but-unguessable. The `token=` param is the only auth — don't post the URL anywhere.

In [ ]:
import subprocess, time, re, secrets, urllib.request, os

if not os.path.exists('/usr/local/bin/cloudflared'):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '/usr/local/bin/cloudflared')
    os.chmod('/usr/local/bin/cloudflared', 0o755)

token = secrets.token_urlsafe(24)
jupyter = subprocess.Popen([
    'jupyter', 'server', '--no-browser', '--ip=0.0.0.0', '--port=8888',
    f'--ServerApp.token={token}', '--ServerApp.password=',
    '--ServerApp.allow_origin=*', '--ServerApp.disable_check_xsrf=True',
])
time.sleep(4)
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8888', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    line = tunnel.stdout.readline().decode('utf-8', 'ignore')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print()
print('Paste this into VS Code (Jupyter: Specify Jupyter Server for Connections):')
print(f'  {url}/?token={token}')